# Automated Dataset Downloader & Setup

This notebook downloads the **Gelar Rasa 2026** competition dataset from Google Drive and extracts it directly into `dataset/raw/`.

> **Note:** Keep your Google Drive link private. Store `GDRIVE_FILE_ID` in your local `.env` file at the project root.

In [ ]:
# 1. Install required packages if not already present
!pip install gdown python-dotenv

In [ ]:
# 2. Imports & Project Root Setup
import os
import re
import zipfile
import gdown
from dotenv import load_dotenv

# Resolve project root (whether notebook is run from root or from notebooks/)
cwd = os.getcwd()
if os.path.basename(cwd) == "notebooks":
    PROJECT_ROOT = os.path.abspath(os.path.join(cwd, ".."))
else:
    PROJECT_ROOT = cwd

print(f"Project Root: {PROJECT_ROOT}")

# Load .env from project root
env_path = os.path.join(PROJECT_ROOT, ".env")
load_dotenv(dotenv_path=env_path)

raw_input_val = os.getenv("GDRIVE_FILE_ID", "").strip()

# Fallback prompt if .env is empty or still has placeholder
if not raw_input_val or raw_input_val == "paste_drive_file_id_here":
    raw_input_val = input("Enter Google Drive File ID or Shareable Link: ").strip()

# Extract File ID if a full Google Drive URL was pasted
match = re.search(r"/d/([a-zA-Z0-9_-]+)", raw_input_val) or re.search(r"id=([a-zA-Z0-9_-]+)", raw_input_val)
FILE_ID = match.group(1) if match else raw_input_val

print(f"Using Google Drive File ID: {FILE_ID[:5]}...{FILE_ID[-5:] if len(FILE_ID) > 10 else ''}")

In [ ]:
# 3. Download Dataset Zip Archive
archive_dir = os.path.join(PROJECT_ROOT, "dataset", "raw")
os.makedirs(archive_dir, exist_ok=True)

zip_destination = os.path.join(archive_dir, "dataset.zip")
download_url = f"https://drive.google.com/uc?id={FILE_ID}"

print(f"Downloading dataset to {zip_destination}...")
gdown.download(download_url, zip_destination, quiet=False)
print("Download completed!")

In [ ]:
# 4. Extract Archive to dataset/raw/
raw_dir = os.path.join(PROJECT_ROOT, "dataset", "raw")
os.makedirs(raw_dir, exist_ok=True)

print(f"Extracting {zip_destination} to {raw_dir}...")
with zipfile.ZipFile(zip_destination, "r") as zip_ref:
    zip_ref.extractall(raw_dir)

print("Extraction complete!")

In [ ]:
# 5. Verification Check
print("=== Dataset Verification ===")
for sub in ["aqms", "health", "master", "metadata"]:
    folder = os.path.join(raw_dir, sub)
    if os.path.exists(folder):
        files = [f for f in os.listdir(folder) if not f.startswith(".")]
        print(f"- dataset/raw/{sub}: {len(files)} files found")
    else:
        print(f"- dataset/raw/{sub}: (Directory not yet populated)")

print("\nAll set! You are ready to run EDA and data pipelines.")